# 🎙️ Sonda Note — GPU Inference Server

Run all 4 cells below. Cell 4 launches the live server and prints your URL.

> ⚡ **Runtime setup:** Go to `Runtime` → `Change runtime type` → select **T4 GPU** before running.
>
> 💡 **Tunneling options:** Defaults to free zero-setup **Cloudflare Quick Tunnels** (`*.trycloudflare.com`). To use a permanent static domain that **never changes**, set your ngrok Authtoken & Static Domain in Cell 4.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q faster-whisper ctranslate2 fastapi 'uvicorn[standard]' python-multipart httpx pyngrok
!apt-get install -qq -y zstd
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import torch
print(f'torch {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
print('Done')

In [ ]:
# ── Cell 2: Start Ollama + pull Qwen2.5:7b ──────────────────────────────────
import subprocess, time, urllib.request
!curl -fsSL https://ollama.com/install.sh | sh
proc = subprocess.Popen(['ollama','serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('Pulling Qwen2.5:7b (~4.5 GB first run)...')
!ollama pull qwen2.5:7b
print('Qwen2.5:7b ready')

In [ ]:
# ── Cell 3: Create server file ────────────────────────────────────────────────
code = '''import json, re, tempfile, os, subprocess, threading, time, contextlib, socket, sys
from pathlib import Path
from typing import Optional, List
import httpx, torch
from fastapi import FastAPI, File, Query, UploadFile, HTTPException
from pydantic import BaseModel
from faster_whisper import WhisperModel

PORT = int(sys.argv[1]) if len(sys.argv) > 1 else 8080
NGROK_DOMAIN = sys.argv[2] if len(sys.argv) > 2 and sys.argv[2] != "none" else None
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[startup] Device: {DEVICE} | Port: {PORT}")
print("[startup] Loading Whisper Large V3...")
asr_model = WhisperModel("large-v3", device=DEVICE, compute_type="int8")
print("[startup] Whisper ready")

@contextlib.asynccontextmanager
async def lifespan(app: FastAPI):
    def run():
        time.sleep(1)
        if NGROK_DOMAIN:
            from pyngrok import ngrok
            url = ngrok.connect(PORT, pyngrok_config=None, domain=NGROK_DOMAIN).public_url
            print("\\n" + "="*64)
            print("  🚀 SONDANOTE GPU SERVER IS LIVE (PERMANENT NGROK DOMAIN)!")
            print(f"  URL: {url}")
            print("="*64)
            print(f"\\n  COLAB_GPU_URL={url}\\n  ASR_PROVIDER=colab\\n  LLM_PROVIDER=colab\\n  OLLAMA_MODEL=qwen2.5:7b\\n")
        else:
            p = subprocess.Popen(["cloudflared","tunnel","--url",f"http://localhost:{PORT}"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in p.stdout:
                m = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", line)
                if m:
                    url = m.group(0)
                    print("\\n" + "="*64)
                    print("  🚀 SONDANOTE GPU SERVER IS LIVE!")
                    print(f"  URL: {url}")
                    print("="*64)
                    print(f"\\n  COLAB_GPU_URL={url}\\n  ASR_PROVIDER=colab\\n  LLM_PROVIDER=colab\\n  OLLAMA_MODEL=qwen2.5:7b\\n")
                    break
    threading.Thread(target=run, daemon=True).start()
    yield

app = FastAPI(title="Sonda Note GPU Inference Server", version="1.0.0", lifespan=lifespan)

@app.get("/health")
def health():
    vram = round(torch.cuda.memory_allocated()/1e9, 1) if DEVICE=="cuda" else 0
    return {"status":"ok","device":DEVICE,"whisper":"large-v3","llm":"qwen2.5:7b","vram_gb":vram}

@app.post("/transcribe")
async def transcribe(file: UploadFile=File(...), language: Optional[str]=Query(None)):
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name
    try:
        segments_gen, info = asr_model.transcribe(tmp_path, language=language or None, beam_size=5, word_timestamps=True)
        segments = list(segments_gen)
        seg_list = [
            {"id":i,"start":round(s.start,3),"end":round(s.end,3),"text":s.text.strip(),
             "words":[{"word":w.word,"start":round(w.start,3),"end":round(w.end,3)} for w in (s.words or [])]}
            for i,s in enumerate(segments) if s.text.strip()
        ]
        duration = segments[-1].end if segments else 0.0
        return {"language":info.language,"duration":round(duration,3),"text":" ".join(s["text"] for s in seg_list),"segments":seg_list}
    finally:
        Path(tmp_path).unlink(missing_ok=True)

class SummariseRequest(BaseModel):
    model: str = "qwen2.5:7b"
    system: str
    prompt: str

@app.post("/summarise")
async def summarise(req: SummariseRequest):
    payload = {"model":req.model,"messages":[{"role":"system","content":req.system},{"role":"user","content":req.prompt}],"stream":False,"format":"json","options":{"temperature":0.2,"num_predict":4096}}
    try:
        async with httpx.AsyncClient(timeout=httpx.Timeout(300.0)) as c:
            resp = await c.post("http://localhost:11434/api/chat", json=payload)
        resp.raise_for_status()
    except Exception as e:
        raise HTTPException(status_code=502, detail=f"Ollama error: {e}")
    raw = resp.json().get("message",{}).get("content","{}")
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"```(?:json)?\\s*(.+?)\\s*```", raw, re.DOTALL)
        parsed = json.loads(m.group(1)) if m else {}
    return {"overview":parsed.get("overview",""),"sections":parsed.get("sections",{}),"action_items":parsed.get("action_items",[]),"insights":parsed.get("insights",[]),"model":req.model}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")
'''
with open('/content/sondanote_server.py', 'w') as f: f.write(code)
print('Server script written. Ready to launch Cell 4.')

In [ ]:
# ── Cell 4: Launch server on dynamic free port ────────────────────────────────
import socket
from pyngrok import ngrok

# (OPTIONAL) Fill these in if you want a free PERMANENT static URL via ngrok!
# Leave blank to default to zero-setup Cloudflare Quick Tunnel (*.trycloudflare.com)
NGROK_AUTHTOKEN = ""       # e.g. "2abc..."
NGROK_STATIC_DOMAIN = ""   # e.g. "sondanote-gpu.ngrok-free.app"

if NGROK_AUTHTOKEN.strip():
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())

def get_free_port():
    for p in [8080, 8088, 8090, 8888, 9000]:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', p)) != 0:
                return p
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        return s.getsockname()[1]

port = get_free_port()
domain_arg = NGROK_STATIC_DOMAIN.strip() if NGROK_STATIC_DOMAIN.strip() else "none"
print(f'Starting server on free port: {port}')
!python3 /content/sondanote_server.py {port} {domain_arg}